# Decision Tree Baseline: Gini vs. Entropy

**Project:** Machine-learning-based recognition of production and changeover activities  
**CRISP-DM phase:** Modelling / Evaluation

## Purpose

This notebook trains **decision trees** for `Label_03` (**1 = Production, 0 = Changeover**) on the 40 features used in notebooks 02 and 03. A decision tree repeatedly asks simple questions of the form "is feature ≤ threshold?" and picks the question that separates the two classes best. It:

1. trains one tree with the **Gini** and one with the **entropy** impurity measure and compares them (AUC, accuracy, F1, precision, recall, confusion matrices, ROC and precision-recall curves);
2. explains the impurity measures with plots and formulas;
3. reads the impurity of every node from the fitted trees and compares the distributions;
4. demonstrates *information gain* on a toy example.

## Workflow

Run notebook 01 first; it creates `normalized_data.csv`. The thesis describes the tree models in Sections 3.2 and 4.2. Notebook 05 continues with restricted trees.

> **Data:** The dataset is public (Zenodo, licence CC BY 4.0) and is **not** included in this repository. See the project README for the download link and citation. This notebook reads `normalized_data.csv`, which notebook 01 creates in the project root. Generated files are written to `../outputs/` and are not committed.

## Important limitations

- **Random split on time-series data.** The split is a stratified *random* split of rows. Neighbouring time steps of a machine are very similar, so near-duplicates can end up in both training and test sets and the scores can be **optimistic**. A chronological or group-wise split is a stricter test.
- **Origin of the 40 features.** The 40 features are the 20 most negative and the 20 most positive Pearson correlations with `Label_03` from notebook 01 (Section 8). That ranking was computed on the whole dataset, including later test rows, so the test scores are slightly biased.
- **Class coding.** Class `1` = Production is the positive class and class `0` = Changeover.
- **Unrestricted trees memorize.** A tree without depth limit can fit the training data perfectly. With a random split, near-identical neighbouring rows in the test set then look "known", so the very high scores should not be read as proof that the model works on new production runs.

## 1. Environment and configuration

Install the dependencies with `pip install -r requirements.txt` (run from the project root). All paths are relative to this notebook, which is expected to live in the `notebooks/` folder.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    PrecisionRecallDisplay, RocCurveDisplay, accuracy_score,
    average_precision_score, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

DATA_PATH = Path("../normalized_data.csv")   # created by notebook 01

LABEL_COL = "Label_03"
SEED = 42
TEST_SIZE = 0.2

## 2. Load the normalized data

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH.resolve()}. "
        "Run notebook 01 first (it creates normalized_data.csv in the project root) "
        "or update DATA_PATH."
    )

df = pd.read_csv(DATA_PATH)
print(f"Loaded: {DATA_PATH}")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

if LABEL_COL not in df.columns:
    raise KeyError(
        f"Target column {LABEL_COL!r} not found. "
        f"The file has {len(df.columns)} columns."
    )

## 3. Select the 40 features and split the data

The same 40 features, the same stratified random 80/20 split and the same seed as in notebook 02. Trees do not need scaling, but they cannot handle missing values in older scikit-learn versions, so missing and infinite values are replaced by the median of the **training** rows.

In [ ]:
feature_list = [
    'Spindle_smoothed_speed',
    'Program_status',
    'Z-axis_position_WCS',
    'Magazine_9998_number_Place_2',
    'Door_open',
    'Z-axis_position_MCS',
    'Magazine_9998_position_Place_3',
    'Program_Idle',
    'Magazine_position_Place_9',
    'Tool_number_Magazine_Place_43',
    'Magazine_number_Place_6',
    'Magazine_number_Place_7',
    'Magazine_number_Place_5',
    'Magazine_9998_position_Place_1',
    'Tool_number_Magazine_Place_9',
    'Magazine_number_Place_8',
    'Tool_magazine_position_WCS',
    'Tool_magazine_position_MCS',
    'Magazine_position_Place_17',
    'Number_of_tools',
    'Feed_override',
    'Rapid_traverse_button',
    'C_Axis_Motor_temperature',
    'Y_Axis_Motor_temperature',
    'Door_status_locked',
    'Spindle_position_WCS',
    'Spindle_position_MCS',
    'X-axis_position_MCS',
    'Net_running_time_program',
    'X_Axis_motor_temperature',
    'B_Axis_Motor_temperature',
    'Program_line_number',
    'Tool_number_Magazine_9998_Place_1',
    'X-axis_position_WCS',
    'Operation_mode',
    'LED_Chip_flushing',
    'Spindle_angular_velocity',
    'Spindle_speed',
    'Spindle_smoothed_modulation_depth',
    'LED_Coolant',
]

missing_features = [c for c in feature_list if c not in df.columns]
if missing_features:
    print("Warning: these features are missing and will be skipped:", missing_features)

feature_list = [c for c in feature_list if c in df.columns]
if not feature_list:
    raise ValueError("None of the expected features were found in the data.")
print(f"Using {len(feature_list)} features")

In [ ]:
X = df[feature_list].replace([np.inf, -np.inf], np.nan)
y = df[LABEL_COL].astype(int).to_numpy()

if set(np.unique(y)) != {0, 1}:
    raise ValueError(
        f"{LABEL_COL} must contain exactly the classes 0 and 1, "
        f"found: {np.unique(y).tolist()}"
    )

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)

# Fill missing values with training medians only (no information from the test set)
train_medians = X_train.median(numeric_only=True)
X_train = X_train.fillna(train_medians)
X_test = X_test.fillna(train_medians)

print(f"Train: {X_train.shape}   Test: {X_test.shape}")

## 4. Train trees with Gini and entropy

Both trees grow without depth limit (`max_depth=None`) and use `class_weight="balanced"` to compensate for the unequal class sizes. **Gini** and **entropy** are two ways to measure how "mixed" the classes in a node are: both are 0 for a pure node and largest for a 50/50 mix.

In [ ]:
dt_gini = DecisionTreeClassifier(
    criterion="gini", class_weight="balanced", random_state=SEED
).fit(X_train, y_train)
dt_entropy = DecisionTreeClassifier(
    criterion="entropy", class_weight="balanced", random_state=SEED
).fit(X_train, y_train)


def eval_table(model, X_eval, y_eval, name):
    proba = model.predict_proba(X_eval)[:, 1]
    pred = (proba >= 0.5).astype(int)
    row = {
        "Model": name,
        "AUC": roc_auc_score(y_eval, proba),
        "AP": average_precision_score(y_eval, proba),
        "ACC": accuracy_score(y_eval, pred),
        "F1": f1_score(y_eval, pred),
        "PREC": precision_score(y_eval, pred, zero_division=0),
        "REC": recall_score(y_eval, pred),
    }
    return row, proba, pred


m_gini, proba_gini, pred_gini = eval_table(dt_gini, X_test, y_test, "DT_gini")
m_ent, proba_ent, pred_ent = eval_table(dt_entropy, X_test, y_test, "DT_entropy")
display(pd.DataFrame([m_gini, m_ent]).round(4))

## 5. Confusion matrices, ROC and precision-recall curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
for ax, name, pred in zip(axes, ["Gini", "Entropy"], [pred_gini, pred_ent]):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Pred 0", "Pred 1"], yticklabels=["True 0", "True 1"], ax=ax)
    ax.set_title(f"Confusion: {name}")
plt.tight_layout()
plt.show()

for model, name in [(dt_gini, "Gini"), (dt_entropy, "Entropy")]:
    RocCurveDisplay.from_estimator(model, X_test, y_test)
    plt.title(f"ROC: {name}")
    plt.tight_layout()
    plt.show()

    PrecisionRecallDisplay.from_estimator(model, X_test, y_test)
    plt.title(f"Precision-Recall: {name}")
    plt.tight_layout()
    plt.show()

## 6. Impurity measures

For a node in which a share `p` of the rows belongs to class 1:

- **Entropy:** `-(p·log2 p + (1-p)·log2(1-p))`
- **Gini:** `2·p·(1-p)`
- **Misclassification error:** `1 - max(p, 1-p)`

All three are 0 for a pure node and largest at `p = 0.5`; entropy reacts more strongly to mixed nodes than Gini.

In [ ]:
def entropy(p):
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return -(p * np.log2(p) + (1 - p) * np.log2(1 - p))


def gini(p):
    return 2 * p * (1 - p)


def misclassification(p):
    return 1 - np.maximum(p, 1 - p)


ps = np.linspace(0.0, 1.0, 201)
plt.figure(figsize=(6, 4))
plt.plot(ps, entropy(ps), label="Entropy")
plt.plot(ps, gini(ps), label="Gini")
plt.plot(ps, misclassification(ps), label="Misclassification")
plt.xlabel("p = share of class 1 in the node")
plt.ylabel("impurity")
plt.title("Impurity measures vs. class proportion")
plt.legend()
plt.tight_layout()
plt.show()

## 7. Node-by-node impurities of the fitted trees

For every node the table shows the number of training rows (`samples`), the class-1 share `p1`, and the three impurities. Note: because of `class_weight="balanced"`, `p1` is the *class-weighted* share, not the raw share.

The sample counts come from `tree_.n_node_samples`. (In scikit-learn 1.4 and newer, `tree_.value` stores class fractions instead of counts, so it cannot be used to count rows.)

In [ ]:
def node_impurities(model, name):
    tree = model.tree_
    values = tree.value.squeeze(axis=1)          # per node: weighted class counts or fractions
    totals = values.sum(axis=1)
    p1 = np.divide(values[:, 1], totals, out=np.zeros_like(totals), where=totals > 0)
    return pd.DataFrame({
        "node_id": np.arange(tree.node_count),
        "samples": tree.n_node_samples,
        "p1": p1,
        "entropy": entropy(p1),
        "gini": gini(p1),
        "misclass": misclassification(p1),
        "is_leaf": tree.children_left == -1,
        "model": name,
    }).sort_values("samples", ascending=False)


nodes_gini = node_impurities(dt_gini, "gini")
nodes_ent = node_impurities(dt_entropy, "entropy")

print(f"Nodes: Gini tree {len(nodes_gini):,}, entropy tree {len(nodes_ent):,}")
display(nodes_gini.head(10))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, col in zip(axes, ["entropy", "gini", "misclass"]):
    sns.kdeplot(nodes_gini[col], label="gini", ax=ax)
    sns.kdeplot(nodes_ent[col], label="entropy", ax=ax)
    ax.set_title(col)
    ax.legend()
plt.tight_layout()
plt.show()

## 8. Toy example: information gain

*Information gain* is how much a split reduces the entropy: parent entropy minus the weighted entropy of the two child nodes. The tree picks, at every node, the feature and threshold with the largest gain. Here the gain is scanned for one feature (the first one in the list) at 31 candidate thresholds.

In [ ]:
feat = feature_list[0]
print("Feature for the demo:", feat)

x = X_train[feat].to_numpy()
H_parent = entropy(y_train.mean())

thresholds = np.quantile(x, np.linspace(0.05, 0.95, 31))
gains = []
for t in thresholds:
    left, right = y_train[x <= t], y_train[x > t]
    if len(left) == 0 or len(right) == 0:
        gains.append((t, 0.0))
        continue
    n = len(y_train)
    H_split = (len(left) / n) * entropy(left.mean()) + (len(right) / n) * entropy(right.mean())
    gains.append((t, H_parent - H_split))

gains = np.array(gains)
best_idx = gains[:, 1].argmax()
best_t, best_gain = gains[best_idx]
print(f"Best threshold by information gain on {feat}: {best_t:.5f} | gain = {best_gain:.5f}")

plt.figure(figsize=(6, 3))
plt.plot(gains[:, 0], gains[:, 1])
plt.axvline(best_t, linestyle="--")
plt.title(f"Information gain vs. threshold on {feat}")
plt.xlabel("threshold")
plt.ylabel("gain (entropy)")
plt.tight_layout()
plt.show()

## 9. Summary and conclusion

Both impurity measures lead to practically identical trees and scores, so the choice between Gini and entropy does not matter here. Both trees grow very large, because nothing limits their depth.

**Reference results from the author's original run** (the cleaned notebook may give slightly different numbers when re-run):

| Model | AUC | Accuracy | F1 | Precision | Recall |
|---|---|---|---|---|---|
| Decision tree, Gini | 0.9939 | 0.9947 | 0.9912 | 0.9908 | 0.9917 |
| Decision tree, entropy | 0.9942 | 0.9949 | 0.9916 | 0.9908 | 0.9923 |

These scores are much higher than the logistic-regression baseline of notebook 02 (F1 = 0.8666), which shows that the classes are separated by non-linear rules. Because of the random split and the unrestricted depth, they are probably optimistic; notebook 05 shows what happens when the depth is restricted.